# 005 · Pseudo-Random Numbers with random

> **Standard Library › Numbers & Math** · Python 3.14

The `random` module generates pseudo-random numbers for simulations, games, sampling, test data and shuffling. It's the counterpart of `System.Random` — with a richer API (weighted choices, sampling, Gaussian values) and a few off-by-one differences that bite C# developers.

**In this notebook you will learn:**

- module-level functions vs independent `random.Random` instances (≈ `Random.Shared` vs `new Random(seed)`)
- seeding for reproducible results
- integers and floats: `randint` (**inclusive** upper bound!), `randrange`, `random`, `uniform`
- picking from collections: `choice`, `choices` with weights, `sample`, `shuffle`
- distributions like `gauss`
- why `random` must **never** be used for passwords or tokens

## .NET mental model

| .NET | Python |
|---|---|
| `Random.Shared` (.NET 6+) | module-level functions: `random.randint(...)` etc. (one hidden shared instance) |
| `new Random(seed)` | `random.Random(seed)` |
| `rng.Next(min, max)` — **max exclusive** | `rng.randrange(min, max)` (exclusive) or `rng.randint(min, max)` (**inclusive**) |
| `rng.NextDouble()` | `rng.random()` |
| `rng.Shuffle(array)` (.NET 8+) | `rng.shuffle(lst)` |
| `rng.GetItems(choices, n)` (.NET 8+) | `rng.choices(population, k=n)` |
| — | `sample`, weighted `choices`, `gauss`, `uniform`, ... |
| `RandomNumberGenerator` (crypto) | `secrets` *(see Standard Library › Numbers & Math › 006 · Secure Randomness with secrets)* |

The generator is the **Mersenne Twister** (MT19937): fast, high quality for simulations, fully deterministic given a seed — and therefore **not** secure.

## Module functions and seeding

```csharp
var rng = new Random(42);       // reproducible
int roll = Random.Shared.Next(1, 7);
```

The module-level functions all use one shared, hidden generator. `random.seed(n)` resets it, so the same seed gives the same sequence on every run.

In [ ]:
import random

random.seed(42)
print(random.random())          # float in [0.0, 1.0) — like NextDouble()
print(random.randint(1, 6))

random.seed(42)                 # same seed...
print(random.random())          # ...same sequence
print(random.randint(1, 6))

## Independent `Random` instances

Seeding the global generator affects **every** caller in the process (including libraries). Prefer your own instance — exactly as you would write `new Random(seed)` in C#. Each instance has its own independent state.

In [ ]:
rng_a = random.Random(7)
rng_b = random.Random(7)
rng_c = random.Random(8)

print([rng_a.randint(1, 6) for _ in range(8)])
print([rng_b.randint(1, 6) for _ in range(8)])   # identical to rng_a
print([rng_c.randint(1, 6) for _ in range(8)])   # different seed, different sequence

**Tip:** pass a `random.Random` instance into functions that need randomness (dependency injection). Tests can then pass a seeded instance for deterministic results.

In [ ]:
def roll_dice(n: int, rng: random.Random) -> list[int]:
    return [rng.randint(1, 6) for _ in range(n)]

print(roll_dice(5, random.Random(123)))
print(roll_dice(5, random.Random(123)))   # reproducible in a unit test

`getstate()` / `setstate()` snapshot and restore a generator mid-sequence:

In [ ]:
rng = random.Random(9)
state = rng.getstate()
first = [rng.random() for _ in range(3)]
rng.setstate(state)                          # rewind
print(first == [rng.random() for _ in range(3)])

## Random integers — mind the upper bound

⚠️ **Gotcha:** `randint(a, b)` includes **both** ends. C#'s `Next(min, max)` excludes `max`. Porting `rng.Next(1, 7)` to `randint(1, 7)` gives you a seven-sided die.

| Python | Range | C# equivalent |
|---|---|---|
| `randint(1, 6)` | 1 … 6 | `Next(1, 7)` |
| `randrange(1, 7)` | 1 … 6 | `Next(1, 7)` |
| `randrange(10)` | 0 … 9 | `Next(10)` |
| `randrange(0, 100, 5)` | 0, 5, … 95 | — |

In [ ]:
from collections import Counter

rng = random.Random(1)
print(sorted(Counter(rng.randint(1, 3) for _ in range(3000))))     # [1, 2, 3] — 3 included
print(sorted(Counter(rng.randrange(1, 3) for _ in range(3000))))   # [1, 2]    — like Next(1, 3)
print([rng.randrange(0, 100, 25) for _ in range(6)])               # multiples of 25 only

## Random floats

In [ ]:
rng = random.Random(2)
print(rng.random())               # [0.0, 1.0)
print(rng.uniform(20.0, 25.0))    # between a and b — e.g. a fake temperature
print(rng.gauss(100, 15))         # normal distribution: mean 100, std dev 15

`gauss(mu, sigma)` (or the thread-safe `normalvariate`) draws from a normal distribution — something `System.Random` doesn't offer. There are many more: `expovariate`, `triangular`, `betavariate`, `lognormvariate`, `binomialvariate` (3.12+), ...

In [ ]:
import statistics

rng = random.Random(5)
heights = [rng.gauss(175, 7) for _ in range(10_000)]
print(round(statistics.mean(heights), 1), round(statistics.stdev(heights), 1))   # close to 175 and 7

## Choosing from collections

| Function | Returns | Replacement? |
|---|---|---|
| `choice(seq)` | one element | — |
| `choices(pop, weights=..., k=n)` | list of `n` | **with** replacement (items can repeat) |
| `sample(pop, k)` | list of `k` unique positions | **without** replacement |
| `shuffle(lst)` | `None` — shuffles **in place** | — |

```csharp
var rng = new Random(3);
string pick = colors[rng.Next(colors.Length)];
string[] picks = rng.GetItems(colors, 5);   // .NET 8+, with replacement
rng.Shuffle(deck);                          // .NET 8+, in place
```

In [ ]:
rng = random.Random(3)
colors = ["red", "green", "blue", "yellow"]

print(rng.choice(colors))           # one item
print(rng.choices(colors, k=5))     # 5 items, repeats allowed — like GetItems
print(rng.sample(range(1, 50), 6))  # 6 unique numbers — a lottery draw

### Weighted choices

`choices` accepts relative `weights=` (or `cum_weights=`), which is how you build loot tables, A/B splits or synthetic test data with realistic proportions:

In [ ]:
rng = random.Random(11)
outcomes = rng.choices(["A", "B", "C"], weights=[5, 3, 2], k=10_000)
print(sorted(Counter(outcomes).items()))      # roughly 50% / 30% / 20%

### Shuffling

`shuffle` mutates the list and returns `None` (like `list.sort()`). To get a shuffled **copy**, use `sample(seq, k=len(seq))`.

In [ ]:
rng = random.Random(3)
deck = ["A", "K", "Q", "J", "10"]
rng.shuffle(deck)
print(deck)

original = ("x", "y", "z")                   # immutable tuple
print(rng.sample(original, k=len(original))) # shuffled copy, original untouched

⚠️ **Gotcha:** errors you may hit:

In [ ]:
for action in (lambda: random.shuffle((1, 2, 3)),   # tuples can't be shuffled in place
               lambda: random.sample([1, 2], 3),    # can't take 3 unique from 2
               lambda: random.choice([])):          # nothing to choose from
    try:
        action()
    except (TypeError, ValueError, IndexError) as e:
        print(f"{type(e).__name__}: {e}")

## Not for security

⚠️ **Gotcha:** the Mersenne Twister is **predictable**: after observing 624 consecutive 32-bit outputs, an attacker can reconstruct the internal state and predict every future value. Never use `random` for passwords, session tokens, password-reset links, API keys or anything an attacker might guess.

Use the `secrets` module instead — it's the equivalent of `System.Security.Cryptography.RandomNumberGenerator` *(see Standard Library › Numbers & Math › 006 · Secure Randomness with secrets)*.

```python
# WRONG for anything security-related
token = "".join(random.choices("abcdef0123456789", k=32))

# RIGHT
import secrets
token = secrets.token_hex(16)
```

**Tip:** `random.SystemRandom()` is a `Random`-compatible class backed by the OS's secure source (`os.urandom`). It's useful if you need a secure version of a method that only exists on `Random` (e.g. `sample`); it ignores seeding.

In [ ]:
sys_rng = random.SystemRandom()
print(len(sys_rng.sample(range(100), 3)))   # values differ every run — so we only print the length

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Shared generator | `Random.Shared` | `random.random()`, `random.randint()`, ... |
| Seeded instance | `new Random(42)` | `random.Random(42)` |
| Int in [a, b) | `rng.Next(a, b)` | `rng.randrange(a, b)` |
| Int in [a, b] | `rng.Next(a, b + 1)` | `rng.randint(a, b)` |
| Float in [0, 1) | `rng.NextDouble()` | `rng.random()` |
| Float in [a, b] | `a + rng.NextDouble() * (b - a)` | `rng.uniform(a, b)` |
| Normal distribution | — | `rng.gauss(mu, sigma)` |
| Pick one | `items[rng.Next(items.Length)]` | `rng.choice(items)` |
| Pick n (with repeats) | `rng.GetItems(items, n)` | `rng.choices(items, k=n)` |
| Weighted pick | hand-written | `rng.choices(items, weights=w, k=n)` |
| Pick n unique | hand-written | `rng.sample(items, n)` |
| Shuffle in place | `rng.Shuffle(array)` | `rng.shuffle(lst)` |
| Secure randomness | `RandomNumberGenerator` | `secrets` module |

## Exercises

1. Simulate 10 000 rolls of two six-sided dice with a seeded `Random` and print how often each total (2–12) occurs.
2. Write `random_order_id(rng)` that returns a string like `"ORD-48213"` (5 digits, 10000–99999). Make it reproducible by passing a seeded generator.
3. A survey picks 3 distinct winners from `["alice", "bob", "carol", "dave", "erin"]`, then gives each of 5 bonus prizes to a random entrant (repeats allowed) where `alice` is twice as likely as anyone else. Use a seeded generator.
4. Estimate π by sampling 100 000 random points in the unit square and counting how many fall inside the quarter circle.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import random
from collections import Counter

# 1 — randint is inclusive, so randint(1, 6) is a fair die
rng = random.Random(2024)
totals = Counter(rng.randint(1, 6) + rng.randint(1, 6) for _ in range(10_000))
for total in range(2, 13):
    print(total, totals[total])

# 2
def random_order_id(rng: random.Random) -> str:
    return f"ORD-{rng.randint(10000, 99999)}"

print(random_order_id(random.Random(1)), random_order_id(random.Random(1)))   # same twice

# 3 — sample: no repeats; choices: repeats + weights
rng = random.Random(7)
people = ["alice", "bob", "carol", "dave", "erin"]
print(rng.sample(people, 3))
print(rng.choices(people, weights=[2, 1, 1, 1, 1], k=5))

# 4 — fraction inside the quarter circle approximates pi / 4
rng = random.Random(0)
n = 100_000
inside = sum(1 for _ in range(n) if rng.random() ** 2 + rng.random() ** 2 <= 1)
print(4 * inside / n)
```

</details>